## 6. Kaggle Alternative

If Colab gives you a bad GPU or kicks you, use Kaggle (30h/week free, 2xT4).

Same flow — just change paths:
```python
DRIVE_ROOT = "/kaggle/working/apexfx"  
# Upload/download archives via Kaggle Datasets
```

In [ ]:
# Show final checkpoint status
os.chdir('/content/apexfx')

result = subprocess.run([
    'python', 'scripts/sync_checkpoints.py', 'status',
    '--checkpoint-dir', CKPT_DIR,
], capture_output=True, text=True)
print(result.stdout)

# Final export
export_path = f"{DRIVE_ROOT}/exports/apexfx_checkpoint_latest.tar.gz"
result = subprocess.run([
    'python', 'scripts/sync_checkpoints.py', 'export',
    '--checkpoint-dir', CKPT_DIR,
    '--output', export_path,
], capture_output=True, text=True)
print(result.stdout)

print("=" * 60)
print("NEXT STEPS:")
print(f"1. Download archive: {export_path}")
print(f"2. Upload to new account's Drive: MyDrive/apexfx_checkpoint.tar.gz")
print(f"3. In new notebook, set:")
print(f'   RESUME = True')
print(f'   IMPORT_ARCHIVE = "/content/drive/MyDrive/apexfx_checkpoint.tar.gz"')
print(f"4. Run all cells")

## 5. Export & Status (run after training stops)

In [ ]:
import threading
import time
import signal

# ─── Auto-export thread ───────────────────────────────────────────
# Periodically exports checkpoint to a portable archive on Drive
# so if Colab kills the session, the archive is already saved

_stop_export = threading.Event()

def auto_export_loop():
    """Background thread: export checkpoint archive every N minutes."""
    export_path = f"{DRIVE_ROOT}/exports/apexfx_checkpoint_latest.tar.gz"
    while not _stop_export.is_set():
        _stop_export.wait(AUTO_EXPORT_MINUTES * 60)
        if _stop_export.is_set():
            break
        try:
            result = subprocess.run([
                'python', '/content/apexfx/scripts/sync_checkpoints.py',
                'export',
                '--checkpoint-dir', CKPT_DIR,
                '--output', export_path,
            ], capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                size_mb = os.path.getsize(export_path) / (1024*1024)
                print(f"\n[AUTO-EXPORT] {time.strftime('%H:%M:%S')} — Checkpoint exported ({size_mb:.1f} MB)")
            else:
                print(f"\n[AUTO-EXPORT] Warning: {result.stdout}")
        except Exception as e:
            print(f"\n[AUTO-EXPORT] Error: {e}")

export_thread = threading.Thread(target=auto_export_loop, daemon=True)
export_thread.start()
print(f"Auto-export enabled: every {AUTO_EXPORT_MINUTES} min -> {DRIVE_ROOT}/exports/")

# ─── Run training ─────────────────────────────────────────────────
os.chdir('/content/apexfx')

cmd = [
    'python', 'scripts/train.py',
    '--config-dir', 'configs/colab',
    '--symbol', SYMBOL,
    '--timeframe', TIMEFRAME,
]
if RESUME:
    cmd.append('--resume')

print(f"\nStarting training: {' '.join(cmd)}")
print("=" * 60)

try:
    process = subprocess.run(cmd, check=False, timeout=43200)  # 12h max
    if process.returncode == 0:
        print("\n[OK] Training completed successfully!")
    else:
        print(f"\n[WARNING] Training exited with code {process.returncode}")
except subprocess.TimeoutExpired:
    print("\n[INFO] 12h timeout reached — session ending")
except KeyboardInterrupt:
    print("\n[INFO] Training interrupted by user")
finally:
    _stop_export.set()
    print("\nFinal export...")
    subprocess.run([
        'python', 'scripts/sync_checkpoints.py',
        'export',
        '--checkpoint-dir', CKPT_DIR,
        '--output', f"{DRIVE_ROOT}/exports/apexfx_checkpoint_latest.tar.gz",
    ], timeout=300)

## 4. Train (with auto-checkpoint & auto-export)

In [ ]:
# Check if data exists
import glob

data_files = glob.glob(f"{DRIVE_ROOT}/data/processed/**/*.parquet", recursive=True)
if data_files:
    print(f"Found {len(data_files)} data file(s):")
    for f in data_files:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")
else:
    print("[WARNING] No data files found!")
    print(f"Upload .parquet files to: {DRIVE_ROOT}/data/processed/")
    print("Or upload from your local machine:")
    print("  from google.colab import files")
    print("  uploaded = files.upload()  # then move to the right dir")

## 3. Upload Data (first time only)

If this is the first account, upload your market data. Subsequent accounts get data from the archive.

**Option A**: Upload parquet files manually to `Google Drive > apexfx > data > processed`

**Option B**: Run data download script (needs MT5 — won't work on Colab, run locally first)

In [ ]:
import sys
sys.path.insert(0, '/content/apexfx')

CKPT_DIR = f"{DRIVE_ROOT}/models/checkpoints"

if IMPORT_ARCHIVE is not None:
    # Import checkpoint from another account's archive
    print(f"Importing checkpoint from: {IMPORT_ARCHIVE}")
    result = subprocess.run([
        'python', '/content/apexfx/scripts/sync_checkpoints.py',
        'import',
        '--archive', IMPORT_ARCHIVE,
        '--checkpoint-dir', CKPT_DIR,
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"[ERROR] {result.stderr}")
    RESUME = True  # Force resume after import

# Show current checkpoint status
result = subprocess.run([
    'python', '/content/apexfx/scripts/sync_checkpoints.py',
    'status',
    '--checkpoint-dir', CKPT_DIR,
], capture_output=True, text=True)
print(result.stdout)

if RESUME:
    print("\n>>> RESUME MODE: Will continue from latest checkpoint")
else:
    print("\n>>> FRESH START: Training from scratch")

## 2. Import Checkpoint (if switching accounts)

In [ ]:
# Clone repo & install
REPO_URL = "https://github.com/YOUR_USERNAME/apexfx.git"  # <-- EDIT THIS

if not os.path.exists('/content/apexfx'):
    # First time: clone
    subprocess.run(['git', 'clone', REPO_URL, '/content/apexfx'], check=True)
else:
    # Already cloned: pull latest
    subprocess.run(['git', '-C', '/content/apexfx', 'pull'], check=True)

# Install dependencies
subprocess.run(['pip', 'install', '-q', '-e', '/content/apexfx'], check=True)

# Verify import
import apexfx
print(f"ApexFX installed successfully")

# Symlink Drive data into project
data_link = '/content/apexfx/data'
if os.path.islink(data_link):
    os.unlink(data_link)
elif os.path.isdir(data_link):
    import shutil
    shutil.rmtree(data_link)
os.symlink(f"{DRIVE_ROOT}/data", data_link)
print(f"Data linked: {data_link} -> {DRIVE_ROOT}/data")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

# Create Drive directories
for d in ['data', 'models/checkpoints', 'logs', 'exports']:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

print(f"Drive mounted. ApexFX root: {DRIVE_ROOT}")

# Check GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

## 1. Mount Drive & Install Dependencies

In [ ]:
# =============================================================
# SETTINGS — EDIT THIS CELL
# =============================================================

# Set True if resuming from a previous session / another account
RESUME = False

# Google Drive folder for checkpoints & data
DRIVE_ROOT = "/content/drive/MyDrive/apexfx"

# If importing checkpoint from another account, put the .tar.gz path here
# Leave None for normal flow (checkpoint already on this Drive)
IMPORT_ARCHIVE = None  # e.g. "/content/drive/MyDrive/apexfx_checkpoint.tar.gz"

# Trading pair to train on
SYMBOL = "EURUSD"
TIMEFRAME = "H1"

# Auto-export checkpoint to a portable archive every N minutes
# (so you can grab it if Colab kills the session)
AUTO_EXPORT_MINUTES = 30

## 0. Settings

# ApexFX Quantum — Colab Training

**Система ротации аккаунтов для бесплатного обучения**

### Схема работы:
```
Аккаунт 1 (12ч) → checkpoint на Drive → скачать архив
                                            ↓
Аккаунт 2 (12ч) → загрузить архив → resume → checkpoint → скачать
                                                              ↓
Аккаунт 3 (12ч) → загрузить → resume → ...
```

### Конфиг (Colab T4 Lite):
| Параметр | Production (4090) | Colab T4 |
|----------|-------------------|----------|
| d_model | 128 | 64 |
| batch_size | 2048 | 512 |
| buffer_size | 2M | 500K |
| Total steps | 5M | 1.15M |
| Est. time | ~8h (4090) | ~30-40h (T4) |

### Инструкция:
1. Запусти все ячейки по порядку
2. Когда сессия заканчивается — checkpoint уже на Drive
3. На новом аккаунте — загрузи архив и запусти с `RESUME = True`